# Bash 5.3 feature deep-dive — GLOBSORT, inherit_errexit, ${| } pipefail, and test builtin changes

Runnable reference for the four behavioral changes introduced in the Bash 5.3 release line. Each section demonstrates the feature in isolation so the effect can be observed, compared, and gated in CI.

**last_verified:** 2026-10-09  
**tool_version:** n/a  
**sources:** []

## Purpose

Provide a focused, executable reference for the four Bash 5.3 features that change runtime behavior in ways existing scripts may not anticipate. This is not a migration checklist — it is a feature-by-feature deep-dive with minimal repros that can be run on any 5.3+ interpreter to confirm the semantics.

## When to use

| Situation | Action |
|---|---|
| Evaluating 5.3 adoption risk | Run each feature cell on the target runtime; record output as the baseline. |
| Writing version-gated code | Wrap 5.3-only syntax in `BASH_VERSINFO` guards using the patterns shown. |
| Debugging a behavioral change | Isolate the feature in question; compare the cell output against the expected semantics below. |

## Prerequisites

- Bash 5.3 or later (`bash --version` must report 5.3.x). Earlier versions will skip 5.3-specific probes.
- Standard utilities: `mktemp`, `printf`, `sort`.
- A scratch directory writable by the current user (cells allocate with `mktemp -d`).

In [ ]:
# Cell 0 — version guard and feature detection
echo "bash: $(bash --version | head -1)"
echo "BASH_VERSINFO: ${BASH_VERSINFO[0]}.${BASH_VERSINFO[1]} patchlevel ${BASH_PATCHLEVEL}"

is_53_or_later=0
if [ "${BASH_VERSINFO[0]}" -eq 5 ] && [ "${BASH_VERSINFO[1]}" -ge 3 ]; then
  is_53_or_later=1
  echo "RUNTIME=5.3+: all feature probes will run"
else
  echo "RUNTIME<5.3: 5.3-specific probes will self-skip"
fi

## 1. GLOBSORT variable

**What it is:** `GLOBSORT` is a **shell variable** (not a `shopt` option) that controls whether pathname expansion results are sorted. When set to a non-empty value, the shell sorts the expanded filenames lexicographically before returning them. When unset or empty, the order is filesystem-dependent (typically directory order).

**Key distinction from 5.2:** In 5.2 and earlier, glob ordering was always filesystem-dependent with no shell-level control. The 5.3 change adds the variable as an opt-in sorting mechanism.

**Common misconception:** `GLOBSORT` is frequently mistaken for a `shopt` option (`shopt -s GLOBSORT`). It is not — `shopt -s GLOBSORT` will report "invalid shell option name". The variable is set with ordinary assignment: `GLOBSORT=1`.

**Scope:** The variable affects all pathname expansions in the current shell context while set. It does not affect brace expansion or other expansions.

In [ ]:
# Cell 1 — GLOBSORT variable demonstration
if [ $is_53_or_later -eq 0 ]; then
  echo "SKIP: GLOBSORT requires Bash 5.3+ (current: $BASH_VERSION)"
  exit 0
fi

g=$(mktemp -d)
touch "$g/10-access.log" "$g/1-access.log" "$g/2-access.log"

echo "--- Unset GLOBSORT (filesystem order) ---"
unset GLOBSORT
(cd "$g" && printf '%s\n' *.log)

echo "--- GLOBSORT=1 (lexicographic sort) ---"
GLOBSORT=1
(cd "$g" && printf '%s\n' *.log)

echo "--- GLOBSORT= (empty, back to filesystem order) ---"
GLOBSORT=
(cd "$g" && printf '%s\n' *.log)

rm -rf "$g"

## 2. inherit_errexit shell option

**What it is:** `inherit_errexit` is a `shopt` option (not a variable) that, when enabled, causes the `errexit` (`set -e`) setting to be inherited by command substitutions (`$(...)`), process substitutions (`<(...)`), and subshells (`(...)`). In Bash 5.2 and earlier, `errexit` was not inherited into these contexts — a failing command inside a substitution would not trigger the shell's exit behavior.

**Behavior change:** With `shopt -s inherit_errexit`, a command substitution that exits non-zero (and is not part of a conditional) will cause the surrounding script to exit, mirroring the behavior of the main command flow.

**Interaction with `set -e` nuances:** The option only takes effect when `errexit` is already enabled (`set -e`). It does not enable `errexit` by itself. The usual `errexit` exemptions still apply (commands in `if`, `&&`/`||` chains, `while`/`until` conditions).

In [ ]:
# Cell 2 — inherit_errexit demonstration
if [ $is_53_or_later -eq 0 ]; then
  echo "SKIP: inherit_errexit requires Bash 5.3+ (current: $BASH_VERSION)"
  exit 0
fi

echo "--- Without inherit_errexit (5.2 behavior) ---"
shopt -u inherit_errexit
set -e
out=$(false; echo "this still runs"); echo "substitution exit: $?"
echo "script continued after failing substitution"
set +e

echo "--- With inherit_errexit (5.3 behavior) ---"
shopt -s inherit_errexit
set -e
echo "About to run failing substitution with inherit_errexit..."
out=$(false; echo "this will not run"); echo "substitution exit: $?"
echo "This line will not be reached"
set +e
shopt -u inherit_errexit

echo "(If you see this, the script continued — likely running on 5.2 or inherit_errexit not supported)"

## 3. ${| } pipefail parameter expansion

**What it is:** The `${| }` expansion (parameter expansion with a pipe character inside the braces) is a new 5.3 feature that exposes the exit status of the last command in a pipeline executed within a command substitution. It is used as `${| command }` or `${| command1 | command2 }`.

**Distinction from `$?`:** After a plain command substitution `$(cmd1 | cmd2)`, `$?` reflects the exit status of the substitution itself (which is the exit status of the last command in the pipeline). The `${| }` form makes this explicit and allows capturing the pipeline status without relying on `$?` after the substitution completes.

**Relationship to `pipefail`:** The `pipefail` shell option (`set -o pipefail`) changes which pipeline exit status is returned (the rightmost non-zero, or zero if all succeed). The `${| }` expansion respects the current `pipefail` setting — it reports the effective pipeline exit status as determined by `pipefail`.

**Syntax note:** The space after `|` is required: `${| cmd }` not `${|cmd}`. The closing brace terminates the expansion.

In [ ]:
# Cell 3 — ${| } pipefail expansion demonstration
if [ $is_53_or_later -eq 0 ]; then
  echo "SKIP: ${| } expansion requires Bash 5.3+ (current: $BASH_VERSION)"
  exit 0
fi

echo "--- Without pipefail (default) ---"
set +o pipefail
result=${| false | true }
echo "pipeline status via ${| }: $result"
echo "$? after expansion: $?"

result=${| true | false }
echo "pipeline status via ${| }: $result"
echo "$? after expansion: $?"

echo "--- With pipefail ---"
set -o pipefail
result=${| false | true }
echo "pipeline status via ${| }: $result"
echo "$? after expansion: $?"

result=${| true | false }
echo "pipeline status via ${| }: $result"
echo "$? after expansion: $?"

set +o pipefail

## 4. test builtin changes

**What changed:** Bash 5.3 tightens the `test`/`[` builtin's argument parsing, particularly for expressions with more than four arguments. In 5.2 and earlier, complex expressions beyond the documented binary/unary forms were parsed with loose precedence rules that could produce surprising results. The 5.3 change aligns the parsing more closely with the POSIX specification for `test`.

**Key behavioral differences:**

- Expressions with 5+ arguments that were previously accepted with implicit precedence now require explicit grouping with parentheses `(` `)` or the `[[ ]]` compound command.
- The `-a` and `-o` binary operators (AND/OR) are deprecated in POSIX and their precedence is ambiguous; 5.3 emits warnings for ambiguous uses and encourages `[[ ]]` with `&&`/`||`.
- Parentheses for grouping must be escaped or quoted when used with `[ ]` (e.g., `\( \)` or `'('` `')'`), but are unquoted in `[[ ]]`.

**Migration guidance:** Prefer `[[ ]]` for complex conditions. If `[ ]` must be used, parenthesize explicitly and test the expression on both 5.2 and 5.3.

In [ ]:
# Cell 4 — test builtin parsing demonstration
if [ $is_53_or_later -eq 0 ]; then
  echo "SKIP: test builtin changes require Bash 5.3+ (current: $BASH_VERSION)"
  exit 0
fi

a=5; b=7; c=3

echo "--- 3-argument form (unchanged) ---"
[ "$a" -gt 0 ] && echo "[ ] simple: ok"
[[ $a -gt 0 ]] && echo "[[ ]] simple: ok"

echo "--- 5-argument form with -a (deprecated) ---"
# In 5.2 this may parse as ([ "$a" -gt 0 ] -a [ "$b" -lt 10 ]) implicitly
# In 5.3 this warns and requires explicit grouping
set +e
if [ "$a" -gt 0 -a "$b" -lt 10 -a "$c" -eq 3 ]; then
  echo "[ ] 5-arg -a chain: ok (exit: $?)"
else
  echo "[ ] 5-arg -a chain: failed (exit: $?)"
fi

echo "--- Explicit grouping with [[ ]] (recommended) ---"
if [[ $a -gt 0 && $b -lt 10 && $c -eq 3 ]]; then
  echo "[[ ]] && chain: ok"
else
  echo "[[ ]] && chain: failed"
fi

echo "--- Explicit grouping with [ ] and chained calls ---"
if [ "$a" -gt 0 ] && [ "$b" -lt 10 ] && [ "$c" -eq 3 ]; then
  echo "[ ] && chain: ok"
else
  echo "[ ] && chain: failed"
fi

echo "--- Parenthesized grouping in [ ] (requires escaping) ---"
if [ \( "$a" -gt 0 -a "$b" -lt 10 \) -a "$c" -eq 3 ]; then
  echo "[ ] with \( \): ok (exit: $?)"
else
  echo "[ ] with \( \): failed (exit: $?)"
fi

## Verify

Gate every feature on:

1. **Interpreter confirmation:** Cell 0 reports `BASH_VERSINFO` major 5, minor >= 3.
2. **Behavioral baseline:** Run cells 1–4 on the target runtime; save outputs as the reference for that patchlevel.
3. **Syntax lint:** `bash -n` on each cell's code; ShellCheck (any recent version) with `-s bash`; review findings.
4. **CI gating:** Wrap 5.3-only cells in the `is_53_or_later` guard from cell 0 so older runners skip cleanly.

## Common errors

- **Treating GLOBSORT as a shopt option:** `shopt -s GLOBSORT` fails with "invalid shell option name". Use `GLOBSORT=1` (variable assignment).
- **Assuming inherit_errexit enables errexit:** It only controls inheritance. `set -e` must be active separately.
- **Omitting the space in `${| }`:** `${|cmd}` is a syntax error. The form is `${| cmd }` with a space after the pipe.
- **Relying on loose test parsing:** 5+ argument `[ ]` expressions with `-a`/`-o` are ambiguous. Use `[[ ]]` with `&&`/`||` or explicit chained `[ ]` calls.
- **Running probes on pre-5.3:** The version guard prints `SKIP` — this is the guard working, not a failure.

## See also

- Bash 5.3 migration checklist: `bash/notebooks/bash-53-migration-checklist.ipynb` (covers patchlevel window validation and production scripting baseline).
- Production scripting patterns: `bash/docs/production-reference-patterns.md` (strict-mode header, validation gates, coproc/idempotency patterns).
- Retry with circuit breaker: `bash/scripts/retry-with-circuit-breaker.sh` (L6 reference implementation).